# Exercise: Spouse Health Contribution Reform (Solution)

## Task
Implement the coming reform of spousal insurance:

- Start from status-quo contribution logic.
- If `old_beitrag == 0` and `bürgergeld__betrag_m_bg == 0`, add:
  - `0.015 * (bruttolohn_m_sn - einnahmen__bruttolohn_m)`
- Otherwise keep status-quo value.

In [ ]:
# Command for installing gettsim (only use on Colab)

# !apt-get update -qq
# !apt-get install -y graphviz graphviz-dev pkg-config
# !pip install pygraphviz
# !pip install gettsim
# !pip install git+https://github.com/ttsim-dev/gettsim-personas.git

In [1]:
import numpy as np
import pandas as pd

# Core GETTSIM API imports:
# - main: run the simulation DAG
# - MainTarget: choose which object to return
# - TTTargets: choose what to compute
# - copy_environment: safe way to create reforms without mutating baseline

from gettsim import InputData, MainTarget, TTTargets, copy_environment, main
from gettsim.tt import policy_function

In [2]:
# Keep the legal regime fixed for baseline vs reform comparability.
POLICY_DATE = "2025-01-01"

## 1) Baseline scenario

In [3]:
from gettsim_personas import einkommensteuer_sozialabgaben

persona = einkommensteuer_sozialabgaben.Couple1Child(policy_date_str=POLICY_DATE)

# One-earner setup: spouse 0 works, spouse 1 has no own wage.
persona_one_earner = persona.upsert_input_data(
    {
        "einnahmen": {"bruttolohn_m": np.array([8000.0, 0.0, 0.0])},
        # Provide this directly to avoid triggering full Bürgergeld computation.
        "bürgergeld": {"betrag_m_bg": np.array([0.0, 0.0, 0.0])},
    }
)

print(persona.description)

Persona to compute income taxes and social insurance contributions. Jointly
        taxed married couple with one child. All transfers are set to zero; don't use
        this persona for low- to mid-income households, as they may be eligible for
        (means-tested) transfers.


In [4]:
tt_targets = TTTargets(
    tree={
        # Focus only on the output affected by the reform.
        "sozialversicherung": {"kranken": {"beitrag": {"betrag_versicherter_m": None}}},
    }
)

In [5]:
# Build baseline policy environment once.
status_quo_environment = main(
    main_target=MainTarget.policy_environment,
    policy_date_str=POLICY_DATE,
    include_warn_nodes=False,
)

# Baseline run under status-quo law and one-earner household input.
baseline = main(
    main_target=MainTarget.results.df_with_nested_columns,
    policy_date_str=POLICY_DATE,
    input_data=InputData.tree(tree=persona_one_earner.input_data_tree),
    tt_targets=tt_targets,
    include_warn_nodes=False,
)

baseline

,sozialversicherung
,kranken
,beitrag
,betrag_versicherter_m
p_id,
0,471.31875
1,0.00000
2,0.00000


## 2) Reform function

In [6]:
# Copy first so baseline environment stays untouched.
reform_environment = copy_environment(status_quo_environment)

@policy_function(start_date="2025-01-01", leaf_name="betrag_versicherter_m")
def betrag_versicherter_m_spouse_reform(
    sozialversicherung__geringfügig_beschäftigt: bool,
    betrag_rentner_m: float,
    betrag_selbstständig_m: float,
    sozialversicherung__in_gleitzone: bool,
    betrag_versicherter_in_gleitzone_m: float,
    betrag_versicherter_regulärer_beitragssatz: float,
    einkommensteuer__einkünfte__ist_hauptberuflich_selbstständig: bool,
    bürgergeld__betrag_m_bg: float,
    einnahmen__bruttolohn_m_sn: float,
    einnahmen__bruttolohn_m: float,
) -> float:
    # Status-quo logic in 2025 (from betrag_versicherter_m_mit_midijob).
    if einkommensteuer__einkünfte__ist_hauptberuflich_selbstständig:
        old_beitrag = betrag_selbstständig_m
    elif sozialversicherung__geringfügig_beschäftigt:
        old_beitrag = 0.0
    elif sozialversicherung__in_gleitzone:
        old_beitrag = betrag_versicherter_in_gleitzone_m
    else:
        old_beitrag = betrag_versicherter_regulärer_beitragssatz

    old_beitrag = old_beitrag + betrag_rentner_m

    # Reform condition and base:
    # - apply only if old own contribution is effectively zero
    # - and if no Bürgergeld is received
    # - base is spouse income: joint SN wage minus own wage
    spouse_income = np.maximum(einnahmen__bruttolohn_m_sn - einnahmen__bruttolohn_m, 0.0)
    apply_surcharge = (np.abs(old_beitrag) < 1e-10) & (bürgergeld__betrag_m_bg == 0)
    surcharge = np.where(apply_surcharge, 0.015 * spouse_income, 0.0)

    return old_beitrag + surcharge

# Inject reform function into copied policy environment.
reform_environment["sozialversicherung"]["kranken"]["beitrag"]["betrag_versicherter_m"] = betrag_versicherter_m_spouse_reform

## 3) Reform run and comparison

In [7]:
# Re-run exactly the same scenario under reformed environment.
reform = main(
    main_target=MainTarget.results.df_with_nested_columns,
    policy_date_str=POLICY_DATE,
    input_data=InputData.tree(tree=persona_one_earner.input_data_tree),
    tt_targets=tt_targets,
    policy_environment=reform_environment,
    include_warn_nodes=False,
)

# Side-by-side comparison of baseline vs reform at person level.
comparison = pd.DataFrame({
    "baseline_kv": baseline[("sozialversicherung", "kranken", "beitrag", "betrag_versicherter_m")],
    "reform_kv": reform[("sozialversicherung", "kranken", "beitrag", "betrag_versicherter_m")],
})
comparison["delta"] = comparison["reform_kv"] - comparison["baseline_kv"]
comparison

,baseline_kv,reform_kv,delta
p_id,,,
0,471.31875,471.31875,0.0
1,0.00000,120.00000,120.0
2,0.00000,0.00000,0.0
